### Load the Anthropic API key

In [58]:
from dotenv import load_dotenv

load_dotenv()

True

### Define the anthropic client and model

In [59]:
from anthropic import Anthropic
client = Anthropic()
model = "claude-sonnet-4-6"

### First API call

In [60]:
message = client.messages.create(
    model=model,
    max_tokens=100,
    messages=[
        {
            "role": "user",
            "content": "hello, my name is Juan David"
        }
    ]
)


In [61]:
message.content[0].text

"Hello, Juan David! It's nice to meet you! 😊\n\nHow are you doing today? Is there something I can help you with?"

### Let's try to have a conversation

In [62]:
message = client.messages.create(
    model=model,
    max_tokens=100,
    messages=[
        {
            "role": "user",
            "content": "What is my name?"
        }
    ]
)

In [63]:
message.content[0].text

"I don't know your name. You haven't shared that information with me. I also don't have access to any personal data about you unless you tell me directly in our conversation. Would you like to introduce yourself?"

### Let's actually build the conversation by storing the whole list of messages

#### Create helper functions

In [ ]:
def add_user_message(messages, text):
    user_message = {
        "role": "user",
        "content": text
    }
    messages.append(user_message)
    return messages

def add_assistant_message(messages, text):
    assistant_message = {
        "role": "assistant",
        "content": text
    }
    messages.append(assistant_message)
    return messages

def chat(messages, system=None, temperature=1.0):

    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system
    message = client.messages.create(
        **params
    )
    return message.content[0].text

#### Set the system for having a conversation

In [65]:
# Initialize messages list

messages = []

messages = add_user_message(messages, "Hi, my name is Juan David")

response_assistant = chat(messages)

messages = add_assistant_message(messages, response_assistant)

messages = add_user_message(messages, "What is my name?")

response_assistant = chat(messages)

messages = add_assistant_message(messages, response_assistant)

In [66]:
messages

[{'role': 'user', 'content': 'Hi, my name is Juan David'},
 {'role': 'assistant',
  'content': 'Hi Juan David! Nice to meet you! 😊 How are you doing? Is there something I can help you with today?'},
 {'role': 'user', 'content': 'What is my name?'},
 {'role': 'assistant',
  'content': 'Your name is **Juan David**! You told me at the beginning of our conversation. 😊 Is there anything else I can help you with?'}]

## Create a chatbot (Uncomment if you want to test it)

In [67]:
# # Get input from the user:

# messages = []

# while True:

#     user_message = input("type something")
#     print(f"user: {user_message}")
#     messages = add_user_message(messages, user_message)
#     assistant_message = chat(messages)
#     print(f"assistant: {assistant_message}")
#     messages = add_assistant_message(messages, assistant_message)

## Now test a system prompt

#### Raw claude call

In [68]:
messages = []

messages = add_user_message(
    messages, "how to solve 3x+2=5"
)

assistant_response = chat(messages)

print(assistant_response)

## Solving 3x + 2 = 5

**Goal:** Isolate x on one side of the equation

### Step 1: Subtract 2 from both sides
$$3x + 2 - 2 = 5 - 2$$
$$3x = 3$$

### Step 2: Divide both sides by 3
$$\frac{3x}{3} = \frac{3}{3}$$
$$x = 1$$

### ✅ Answer: x = 1

### Check your answer:
Substitute x = 1 back into the original equation:
- 3(1) + 2 = 5
- 3 + 2 = 5 ✓


#### Adding a system prompt acting as a tutor

In [69]:
system_prompt = """
You're math tutor, only give hints to the student. Do not
give the answer to the student, guide him.
"""

messages = []

messages = add_user_message(messages, "how to solve 3x+2=5")

assistant_response = chat(messages, system=system_prompt)

print(assistant_response)

Great question! Let's work through this step by step. I'll give you a hint:

**Hint 1:** Your goal is to get **x by itself** on one side of the equation.

Think about this: what if you first tried to **get rid of the +2** on the left side? 

What operation could you do to **both sides** of the equation to remove it? 🤔


## Exercise

#### Raw claude call

In [70]:
messages = []

messages = add_user_message(
    messages,
    "Write a Python function that checks a string for duplicate characters"
)

assistant_response = chat(messages)

print(assistant_response)

## Duplicate Character Checker

Here's a Python function that checks a string for duplicate characters:

```python
def check_duplicates(string: str) -> dict:
    """
    Check a string for duplicate characters.
    
    Args:
        string: The input string to check.
    
    Returns:
        A dictionary containing:
        - 'has_duplicates': Boolean indicating if duplicates exist
        - 'duplicates': Dictionary of duplicate characters and their counts
        - 'unique_chars': List of characters that appear only once
    """
    char_count = {}

    # Count occurrences of each character
    for char in string:
        char_count[char] = char_count.get(char, 0) + 1

    # Separate duplicates from unique characters
    duplicates = {char: count for char, count in char_count.items() if count > 1}
    unique_chars = [char for char, count in char_count.items() if count == 1]

    return {
        "has_duplicates": len(duplicates) > 0,
        "duplicates": duplicates,
        "unique

#### Now with system prompt

In [71]:
system_prompt = """
You're a staff software engineer that answers as concisely as possible, only give the code to the user, do not explain anything.
"""

messages = []

messages = add_user_message(
    messages,
    "Write a Python function that checks a string for duplicate characters"
)

assistant_response = chat(messages, system=system_prompt)

print(assistant_response)

```python
def has_duplicate_chars(s: str) -> bool:
    return len(s) != len(set(s))
```


## Let's experiment with temperature

In [72]:
messages = []

messages = add_user_message(messages, "Generate a one sentence movie idea")

answer = chat(messages, temperature=1)

print(answer)

Here's a movie idea:

**A retired safecracker with early-onset dementia must break into a bank vault one last time before he forgets how — and before he forgets why.**


In [73]:
messages = []

messages = add_user_message(messages, "Generate a one sentence movie idea")

answer = chat(messages, temperature=0)

print(answer)

Here's a movie idea:

**A retired safecracker with early-onset Alzheimer's must pull off one final heist before he forgets the combination he memorized decades ago — the only thing that can prove his son's innocence.**


In [74]:
messages = []

messages = add_user_message(messages, "Generate a one sentence movie idea")

answer = chat(messages, temperature=0)

print(answer)

Here's a movie idea:

**A retired safecracker with early-onset Alzheimer's must pull off one final heist before he forgets the combination to the vault that holds the only evidence to free his wrongfully imprisoned son.**


## Streaming

In [76]:
messages = []

messages = add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

In [ ]:
for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_01VDtzueHpzAxrymcXvmigUA', container=None, content=[], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=18, output_tokens=1, output_tokens_details=None, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='Here', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' is a one sentence description of a fake database:\n\n**"Nova', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='Base is a cl

## Simpler streaming with claude python SDK

In [87]:
messages = []

messages = add_user_message(messages, "Write a one sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages,
    temperature=0
) as stream:
    for text in stream.text_stream:
        pass

    final_message = stream.get_final_message()